# TIMs REST/Python API

Based on [ESRI ArcGIS API](https://developers.arcgis.com/rest/services-reference/enterprise/get-started-with-the-services-directory.htm)

In [13]:
from IPython.core.display import HTML
from bs4 import BeautifulSoup
import requests
import json
import re

# Historic Data Query (Past 4 Years)

In [14]:
full_data_url = "https://gis.dot.state.oh.us/arcgis/rest/services/TIMS/Projects/MapServer/4/query?where=1%3D1&text=&objectIds=&time=&geometry=&geometryType=esriGeometryEnvelope&inSR=&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&returnTrueCurves=false&maxAllowableOffset=&geometryPrecision=&outSR=&having=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&outStatistics=&returnZ=false&returnM=false&gdbVersion=&historicMoment=&returnDistinctValues=false&resultOffset=&resultRecordCount=&queryByDistance=&returnExtentOnly=false&datumTransformation=&parameterValues=&rangeValues=&quantizationParameters=&featureEncoding=esriDefault&f=pjson"

page = requests.get(full_data_url)
data_json = json.loads(page.content)

In [15]:
from munch import Munch
project_list = []

In [16]:
for project in data_json['features']:
    project_obj = Munch(project['attributes'])
    project_list.append(project_obj)

In [17]:
# Pull this cookie
project_list[3].PROJECT_PLANS_URL

'http://contracts.dot.state.oh.us/search.jsp?cabinetId=1002&PID_NUM=110375'

In [27]:
headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Cookie": "_ga=GA1.3.1674095645.1710160295; _ga_H9WHB924EN=GS1.3.1710509539.2.0.1710509539.0.0.0; _ga_CCD0M9E21K=GS1.3.1713199631.5.1.1713199647.0.0.0; JSESSIONID=eC1nJgZu6uzzgSeUeqbrwJQhHiuY9nZEaOdeBw8Y.dotidpxep02",
    "Host": "contracts.dot.state.oh.us",
    "Referer": "https://contracts.dot.state.oh.us/document/getDocumentProperties.do?documentId=974043&cabinetId=1002&context=documentSearch&tabId=-618cef00&displayMode=condensed&from=topNav&actionForm=documentSearchCriteriaForm&actionMode=display"
    }

In [28]:
len(project_list)

5336

In [29]:
for index, item in enumerate(project_list):
    if item.PID_NBR == 113626:
        print(index)
        break
else:
    index = -1

383


In [30]:
project_list[0]

Munch({'ObjectID': 3, 'GIS_ID': 57043, 'PID_NBR': 104612, 'DISTRICT_NBR': 4, 'LOCALE_SHORT_NME': 'TRU', 'COUNTY_NME': 'Trumbull', 'PROJECT_NME': 'TRU Reserve Ave Bridges', 'CONTRACT_TYPE': 'Standard Build', 'PRIMARY_FUND_CATEGORY_TXT': 'CRRSAA Highway Large MPOs', 'PROJECT_MANAGER_NME': 'SURMA, CHRISTINE M', 'RESERVOIR_YEAR': None, 'TIER': None, 'ODOT_LETTING': 'Local Let', 'SCHEDULE_TYPE_SHORT_NME': None, 'ENV_PROJECT_MANAGER_NME': 'CARPENTER, SEAN D', 'AREA_ENGINEER_NME': 'VICKERS, VICKI  ', 'PROJECT_ENGINEER_NME': 'PATMAN, GERALDINE E', 'DESIGN_AGENCY': 'Warren, City of', 'SPONSORING_AGENCY': 'Warren, City of', 'PDP_SHORT_NAME': 'Path 2', 'PRIMARY_WORK_CATEGORY': 'Bridge Preservation', 'PROJECT_STATUS': 'Awarded', 'FISCAL_YEAR': '2023', 'INHOUSE_DESIGN_FULL_NME': None, 'EST_TOTAL_CONSTR_COST': 1807013.45, 'STATE_PROJECT_NBR': '22N178', 'CONSTR_VENDOR_NME': 'B G Trucking & Construction, Inc.', 'STIP_FLAG': 'Y', 'CURRENT_STIP_CO_AMT': 800000.0, 'PROJECT_PLANS_URL': 'http://contracts.d

In [ ]:
i = 762
while i <= len(project_list):
    print(i)
    
    # Call the URL Pulled from tims
    r = requests.get(project_list[i].PROJECT_PLANS_URL)
    soup = BeautifulSoup(r.content, 'html.parser')

    # Extract the Attributes for the project from the digital paper URL
    try:
        doc_info = [elm['href'] for elm in soup.find_all("a") if elm['href'][:37]=='javascript:stampCheckForThumbnailLink'][0]
        download_url = re.findall(r"'(.*?)'", doc_info)[0]

        project_attributes_temp_list = download_url.split('?')[1].split('&')
        att_dict = dict([x.split('=') for x in project_attributes_temp_list])

        print(att_dict)

        url = f"https://contracts.dot.state.oh.us/" + download_url
        print(url)
        
        r = requests.get(url, headers=headers)  # to get content after redirection

        with open(f'output/ODOT_plans/{project_list[i].PID_NBR}.pdf', 'wb') as f:
            f.write(r.content)
    except:
        print('Error')
    i += 1